Compare os métodos da bisseção, falsa posição e do ponto fixo, Newton-Rhapson e da Secante localizando a raiz $x^{*}$ das seguintes equações:

Para as avaliações, deve-se considerar:

* O número máximo de iterações de todos os métodos testados não pode ultrapassar 200;
* A tolerância deve ser de $10^{-10}$;
* Para os métodos abertos, escolha um dos limites do intervalo.

Para cada método, estamos interessados em comparar:
* Raiz
* Número de iterações até o critério de parada
* Se houve erro de convergência
* tempo de cálculo (procure como calcular tempo de execução usando jupyter notebooks, como %timeit).

In [ ]:
import numpy as np
import time

# --- 1. Método da Bisseção ---
def metodo_bissecao(f, a, b, tol, max_iter=200):
    start_time = time.time()
    iteracoes = 0
    raiz = 0
    convergiu = False

    if f(a) * f(b) >= 0:
        return None, 0, False, 0.0

    for i in range(1, max_iter + 1):
        c = (a + b) / 2
        fc = f(c)

        if (b - a) / 2 < tol or abs(fc) < tol:
            raiz = c
            iteracoes = i
            convergiu = True
            break

        if f(a) * fc < 0:
            b = c
        else:
            a = c

        iteracoes = i
        raiz = c

    end_time = time.time()
    return raiz, iteracoes, convergiu, end_time - start_time

# --- 2. Método do Ponto Fixo ---
def metodo_ponto_fixo(g, x0, tol, max_iter=200):
    start_time = time.time()
    iteracoes = 0
    x_ant = x0
    convergiu = False

    for i in range(1, max_iter + 1):
        x_novo = g(x_ant)

        erro = 0
        if x_novo != 0:
            erro = np.abs((x_novo - x_ant) / x_novo) * 100

        if erro <= tol or abs(x_novo - x_ant) < tol: # Critério misto para robustez
            convergiu = True
            iteracoes = i
            return x_novo, iteracoes, convergiu, time.time() - start_time

        x_ant = x_novo
        iteracoes = i

    return x_ant, iteracoes, convergiu, time.time() - start_time

# --- 3. Método de Newton-Raphson ---
def metodo_newton(f, df, x0, tol, max_iter=200):
    start_time = time.time()
    x_ant = x0
    iteracoes = 0
    convergiu = False

    for i in range(1, max_iter + 1):
        derivada = df(x_ant)

        if derivada == 0: # Evita divisão por zero
            break

        x_novo = x_ant - (f(x_ant) / derivada)

        if abs(x_novo - x_ant) < tol:
            convergiu = True
            iteracoes = i
            return x_novo, iteracoes, convergiu, time.time() - start_time

        x_ant = x_novo
        iteracoes = i

    return x_ant, iteracoes, convergiu, time.time() - start_time

# --- 4. Método da Secante ---
def metodo_secante(f, x0, x1, tol, max_iter=200):
    start_time = time.time()
    iteracoes = 0
    convergiu = False

    for i in range(1, max_iter + 1):
        fx0 = f(x0)
        fx1 = f(x1)

        if (fx1 - fx0) == 0: # Evita divisão por zero
            break

        # Fórmula da Secante
        x_novo = x1 - (fx1 * (x1 - x0)) / (fx1 - fx0)

        if abs(x_novo - x1) < tol:
            convergiu = True
            iteracoes = i
            return x_novo, iteracoes, convergiu, time.time() - start_time

        x0 = x1
        x1 = x_novo
        iteracoes = i

    return x1, iteracoes, convergiu, time.time() - start_time

# --- Função de Comparação Atualizada ---
def comparar_metodos(f, df, g, a, b, x0, tol=1e-10, max_iter=200):
    """
    f: Função f(x) = 0
    df: Derivada f'(x) (para Newton)
    g: Função de iteração x = g(x) (para Ponto Fixo)
    a, b: Intervalo (para Bisseção e pontos iniciais da Secante)
    x0: Chute inicial (para Newton e Ponto Fixo)
    """
    print(f"{'Metodo':<15} {'Raiz':<15} {'Iter':<10} {'Tempo(s)':<15} {'Conv'}")
    print("-" * 65)

    # 1. Bisseção
    r, it, conv, t = metodo_bissecao(f, a, b, tol, max_iter)
    r_str = f"{r:.8f}" if r is not None else "Falha"
    print(f"{'Bissecao':<15} {r_str:<15} {it:<10} {t:<15.6f} {conv}")

    # 2. Ponto Fixo
    r, it, conv, t = metodo_ponto_fixo(g, x0, tol, max_iter)
    print(f"{'Ponto Fixo':<15} {r:<15.8f} {it:<10} {t:<15.6f} {conv}")

    # 3. Newton
    r, it, conv, t = metodo_newton(f, df, x0, tol, max_iter)
    print(f"{'Newton':<15} {r:<15.8f} {it:<10} {t:<15.6f} {conv}")

    # 4. Secante (Usando a e b como os dois pontos iniciais)
    r, it, conv, t = metodo_secante(f, a, b, tol, max_iter)
    print(f"{'Secante':<15} {r:<15.8f} {it:<10} {t:<15.6f} {conv}")

(a) $f_{1}(x)=2x^{4}+4x^{3}+3x^{2}-10x-15$, com $x^{*}\in[0,3]$

In [ ]:
def f1(x):
    return 2*x**4 + 4*x**3 + 3*x**2 - 10*x - 15

def df1(x):
    # Derivada para o método de Newton: 8x³ + 12x² + 6x - 10
    return 8*x**3 + 12*x**2 + 6*x - 10

def g1(x):
    # Ponto fixo com relaxamento (alpha=60) para estabilidade
    return x - f1(x)/60

# Teste com os 4 métodos
# Intervalo [0, 3], Chute inicial x0 = 1.5
comparar_metodos(f1, df1, g1, 0, 3, 1.5)

Metodo          Raiz            Iter       Tempo(s)        Conv
-----------------------------------------------------------------
Bissecao        1.49287871      35         0.000079        True
Ponto Fixo      1.49287871      10         0.000087        True
Newton          1.49287871      4          0.000006        True
Secante         -1.30038413     11         0.000028        True


(b) $f_{2}(x)=(x+3)(x+1)(x-2)^{3},$ com $x^{*}\in[0,5]$

In [ ]:
def f2(x):
    return (x+3)*(x+1)*(x-2)**3

def df2(x):
    # Derivada pela regra do produto para f(x) = (x+3)(x+1)(x-2)³
    # f'(x) = 1*(x+1)(x-2)³ + (x+3)*1*(x-2)³ + (x+3)(x+1)*3(x-2)²
    term1 = (x+1) * (x-2)**3
    term2 = (x+3) * (x-2)**3
    term3 = (x+3) * (x+1) * 3 * (x-2)**2
    return term1 + term2 + term3

def g2(x):
    # Ponto fixo com fator de relaxamento (alpha=100) devido ao grau 5
    return x - f2(x)/100

# Teste com os 4 métodos
# Intervalo [0, 5], Chute inicial x0 = 2.5
comparar_metodos(f2, df2, g2, 0, 5, 2.5)

Metodo          Raiz            Iter       Tempo(s)        Conv
-----------------------------------------------------------------
Bissecao        2.00012207      13         0.000036        True
Ponto Fixo      2.11870631      200        0.000408        False
Newton          2.00000000      54         0.000058        True
Secante         2.00000000      78         0.000054        True


(c) $f_{3}(x)=5x^{3}+x^{2}-e^{1-2x}+\cos(x)+20$, com $x^{*}\in[-5,5]$

In [ ]:
def f3(x):
    return 5*x**3 + x**2 - np.exp(1-2*x) + np.cos(x) + 20

def df3(x):
    # Derivada: 15x² + 2x + 2*exp(1-2x) - sin(x)
    return 15*x**2 + 2*x + 2*np.exp(1-2*x) - np.sin(x)

def g3(x):
    # Isolando x³ para convergência (Raiz cúbica)
    return np.cbrt((-x**2 + np.exp(1-2*x) - np.cos(x) - 20) / 5)

# Teste com os 4 métodos
# Intervalo [-5, 5], Chute inicial x0 = -1.0
comparar_metodos(f3, df3, g3, -5, 5, -1.0)

Metodo          Raiz            Iter       Tempo(s)        Conv
-----------------------------------------------------------------
Bissecao        -0.92956046     37         0.000417        True
Ponto Fixo      -1.69789017     200        0.001276        False
Newton          -0.92956046     5          0.000046        True
Secante         -0.92956046     22         0.000191        True


(d) $f_{4}(x)=\sin(x)x+4$ com $x^{*}\in[1,5]$

In [ ]:
def f4(x):
    return np.sin(x)*x + 4

def df4(x):
    # Derivada do produto: f'(x) = sin(x) + x*cos(x)
    return np.sin(x) + x*np.cos(x)

def g4(x):
    # Ponto fixo com relaxamento (alpha=5) para garantir convergência
    return x + (np.sin(x)*x + 4)/5

# Teste com os 4 métodos
# Intervalo [1, 5], Chute inicial x0 = 4.0
comparar_metodos(f4, df4, g4, 1, 5, 4.0)

Metodo          Raiz            Iter       Tempo(s)        Conv
-----------------------------------------------------------------
Bissecao        4.32323954      34         0.000334        True
Ponto Fixo      4.32323954      31         0.000109        True
Newton          4.32323954      5          0.000028        True
Secante         4.32323954      8          0.000038        True


(e) $f_{5}(x)=(x-3)^{5}\ln(x)$ com $x^{*}\in[2,5]$

In [ ]:
def f5(x):
    return (x-3)**5 * np.log(x)

def df5(x):
    # Derivada pela regra do produto:
    # f'(x) = 5(x-3)^4 * ln(x) + (x-3)^5 * (1/x)
    return 5*(x-3)**4 * np.log(x) + ((x-3)**5 / x)

def g5(x):
    # Método de relaxamento (divisor 10) para tentar estabilizar a convergência
    return x - ((x-3)**5 * np.log(x))/10

# Teste com os 4 métodos
# Intervalo [2, 5], Chute inicial x0 = 4.0
comparar_metodos(f5, df5, g5, 2, 5, 4.0)

Metodo          Raiz            Iter       Tempo(s)        Conv
-----------------------------------------------------------------
Bissecao        3.00781250      7          0.000116        True
Ponto Fixo      3.31573539      200        0.000684        False
Newton          3.00000000      98         0.000348        True
Secante         3.00000000      138        0.000486        True


(f) $f_{6}(x)=x^{10}-1$, com $x^{*}\in[0.8,1.2]$

In [ ]:
def f6(x):
    return x**10 - 1

def df6(x):
    # Derivada: 10x^9
    return 10 * x**9

def g6(x):
    # A derivada no ponto x=1 é 10.
    # Usamos divisor 12 para garantir que |g'(x)| < 1 e o método convirja.
    return x - (x**10 - 1)/12

# Teste com os 4 métodos
# Intervalo [0.8, 1.2], Chute inicial x0 = 1.1
comparar_metodos(f6, df6, g6, 0.8, 1.2, 1.1)

Metodo          Raiz            Iter       Tempo(s)        Conv
-----------------------------------------------------------------
Bissecao        1.00000000      1          0.000010        True
Ponto Fixo      1.00000000      14         0.000080        True
Newton          1.00000000      6          0.000008        True
Secante         1.00000000      11         0.000014        True
